In [0]:
%pip install yfinance

In [0]:
import json
import time
import math
from datetime import datetime
import yfinance as yf

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.default;
CREATE VOLUME IF NOT EXISTS workspace.default.raw_landing;

In [0]:
# Create landing directory and configure ticker mapping
dbutils.fs.mkdirs("/Volumes/workspace/default/raw_landing/ticks")

# Must match the ticker order used in your dim_asset seed table
TICKERS = ["AAPL", "MSFT", "GOOGL", "NVDA", "AMZN"]

# Map ticker symbols to AST-100x asset IDs
ticker_map = {
    ticker: f"AST-{1000 + idx}" for idx, ticker in enumerate(TICKERS)
}

print("Initialized Ticker Mapping:", ticker_map)

In [0]:
while True:
    live_ticks = []
    # Fetch 1-minute interval data for latest tick
    df_data = yf.download(TICKERS, period="1d", interval="1m")["Close"].iloc[-1]

    for ticker in TICKERS:
        price = df_data[ticker] if len(TICKERS) > 1 else df_data
        if not math.isnan(price):
            live_ticks.append({
                "asset_id": ticker_map[ticker],
                "price_date": str(datetime.now().date()),
                "open_price": float(price),
                "close_price": float(price),
                "adj_close": float(price),
                "volume": 0,
                "ingested_at": str(datetime.now()),
            })

    # Dump tick array to landing volume folder as JSON
    file_time = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_path = f"/Volumes/workspace/default/raw_landing/ticks/tick_{file_time}.json"
    dbutils.fs.put(file_path, json.dumps(live_ticks), overwrite=True)

    time.sleep(60)  # Wait 1 minute before next tick